# Project D — Consolidated baseline vs depth-C (Tasks 1a and 1b)

This notebook reruns exactly two configurations:

- **baseline**: `LinearSVC` with fixed `C=1.0` on all depths.
- **depthC**: `LinearSVC` with depth-specific `C` from phase-3 results:
  - depth 0: `C=0.1`
  - depth 1: `C=10`
  - depth 2: `C=10`
  - depth 3: `C=10`
  - fallback for other depths: `C=1.0`

It reports both assignment tasks:
- **Task 1a**: H1-only metrics (`h1_*`) via `train_and_summarize`.
- **Task 1b**: full-tree metrics (`ft_*`, `leaf_*`) via `train_full_tree_and_summarize`.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path
import pandas as pd
from IPython.display import display

from topic_hierarchy import load_topic_tree
from hierarchical_training_data import make_multilabel_binary_pool_data
from hierarchical_classifier import (
    svm_linear_binary_edge_factory,
    linear_svc_estimator_by_depth,
    binary_edge_factory,
)
from hierarchical_summary_metrics import train_and_summarize, train_full_tree_and_summarize


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / 'topics.csv').exists():
        return here
    for p in [here, *here.parents]:
        if (p / 'topics.csv').exists():
            return p
    raise FileNotFoundError('topics.csv not found — set cwd to Homework 4')


BASE = homework4_base()
tree = load_topic_tree(str(BASE / 'topics.csv'))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))

SPLIT = 'test'
RESTRICT = True
MAX_FEATURES = 8000
VERBOSE_1B = False

DEPTH_C_MAP = {0: 0.1, 1: 10.0, 2: 10.0, 3: 10.0}

print('BASE', BASE)
print('train', len(mldata.train_ids()), 'test', len(mldata.test_ids()))
print('depth-C map', DEPTH_C_MAP)


### Build the two factories


In [ ]:
def make_baseline_factory():
    return svm_linear_binary_edge_factory(
        max_features=MAX_FEATURES,
        text_vectorizer='tfidf',
        svm_kw=dict(C=1.0, dual=False, max_iter=8000),
    )


def make_depthc_factory():
    est = linear_svc_estimator_by_depth(
        DEPTH_C_MAP,
        default_c=1.0,
        dual=False,
        max_iter=8000,
    )
    return binary_edge_factory(
        tfidf_kw=dict(min_df=2, max_features=MAX_FEATURES),
        estimator=est,
        text_vectorizer='tfidf',
    )


EXPERIMENTS = [
    ('baseline', make_baseline_factory),
    ('depthC', make_depthc_factory),
]


### Task 1a rerun (H1 only)


In [ ]:
rows_1a = []
for name, fac_fn in EXPERIMENTS:
    print('
' + '=' * 72)
    print(f'TASK 1A [{name}]')
    print('=' * 72)
    t0 = time.perf_counter()
    _, row = train_and_summarize(
        name,
        tree,
        mldata,
        fac_fn(),
        split=SPLIT,
        include_leaf=False,
        restrict_to_parent_subtree=RESTRICT,
    )
    wall = time.perf_counter() - t0
    rows_1a.append({
        'task': '1a',
        'config': name,
        'h1_macro_f1': row.get('h1_macro_f1'),
        'h1_pooled_micro_f1': row.get('h1_pooled_micro_f1'),
        'wall_time_sec': wall,
    })
    print(
        f"ok h1_macro_f1={row.get('h1_macro_f1', float('nan')):.4f} "
        f"h1_pooled_micro_f1={row.get('h1_pooled_micro_f1', float('nan')):.4f} "
        f"({wall:.1f}s)"
    )

df_1a = pd.DataFrame(rows_1a)
if not df_1a.empty:
    b = float(df_1a.loc[df_1a['config'] == 'baseline', 'h1_pooled_micro_f1'].iloc[0])
    df_1a['delta_vs_baseline_h1_pooled_micro_f1'] = df_1a['h1_pooled_micro_f1'] - b
    display(df_1a.round(4))


### Task 1b rerun (full tree)

This is the slow part. We only run the two configurations above.


In [ ]:
rows_1b = []
for name, fac_fn in EXPERIMENTS:
    print('
' + '=' * 72)
    print(f'TASK 1B [{name}]')
    print('=' * 72)
    t0 = time.perf_counter()
    _, row = train_full_tree_and_summarize(
        name,
        tree,
        mldata,
        fac_fn(),
        split=SPLIT,
        verbose=VERBOSE_1B,
        restrict_to_parent_subtree=RESTRICT,
    )
    wall = time.perf_counter() - t0
    rows_1b.append({
        'task': '1b',
        'config': name,
        'ft_pooled_micro_f1': row.get('ft_pooled_micro_f1'),
        'ft_macro_f1': row.get('ft_macro_f1'),
        'leaf_micro_f1': row.get('leaf_micro_f1'),
        'path_gold_branch_recall': row.get('path_gold_branch_recall'),
        'wall_time_sec': wall,
    })
    print(
        f"ok ft_pooled_micro_f1={row.get('ft_pooled_micro_f1', float('nan')):.4f} "
        f"leaf_micro_f1={row.get('leaf_micro_f1', float('nan')):.4f} ({wall:.1f}s)"
    )

df_1b = pd.DataFrame(rows_1b)
if not df_1b.empty:
    b_ft = float(df_1b.loc[df_1b['config'] == 'baseline', 'ft_pooled_micro_f1'].iloc[0])
    b_leaf = float(df_1b.loc[df_1b['config'] == 'baseline', 'leaf_micro_f1'].iloc[0])
    df_1b['delta_vs_baseline_ft_pooled_micro_f1'] = df_1b['ft_pooled_micro_f1'] - b_ft
    df_1b['delta_vs_baseline_leaf_micro_f1'] = df_1b['leaf_micro_f1'] - b_leaf
    display(df_1b.round(4))


### Consolidated output


In [ ]:
final_rows = []
if 'df_1a' in globals() and not df_1a.empty:
    final_rows.append(df_1a)
if 'df_1b' in globals() and not df_1b.empty:
    final_rows.append(df_1b)

consolidated = pd.concat(final_rows, ignore_index=True) if final_rows else pd.DataFrame()
display(consolidated.round(4))

out_csv = BASE / 'projectD_depthC_consolidated_results.csv'
if not consolidated.empty:
    consolidated.to_csv(out_csv, index=False)
    print('Wrote', out_csv)
else:
    print('No results to save')
